# Skin Cancer Detection — impacto do pré-processamento no EfficientNetB4 (HAM10000)

**Objetivo do estudo:** analisar como diferentes **métodos de pré-processamento** das imagens dermatoscópicas (remoção de pelos, correção de cor, realce de contraste, recortes, etc.) alteram o desempenho de um EfficientNetB4 na classificação de 7 tipos de lesões de pele.

**Desenho em resumo**
1. Uma linha de base simples (redimensionar + RGB) e um CNN básico como referência.
2. Uma **configuração de treino fixa** (escolhida com experiências de controlo), para que a única coisa a variar seja o pré-processamento.
3. Cada método de pré-processamento é um **passo composável** (`'hair'`, `'cc'`, `'clahe'`, ...). Basta indicá-lo em `run_experiment(..., steps=(...))`.
4. Comparação por **macro F1**, **macro recall** e **recall de `mel`**, com repetições (*seeds*) para separar efeitos reais de ruído.

**Índice**
1. Instalação, imports e configuração
2. Dataset (organização em pastas e classes)
3. Distribuição das classes
4. Exemplos de imagens
5. Pré-processamento: passos, carregamento e visualização
6. Funções auxiliares (gráficos, métricas, registo de resultados)
7. Baseline — CNN básico
8. EfficientNet — construção, treino e fine-tuning
9. Experiências (controlo, pré-processamento, repetições)
10. Comparação de resultados
11. Análise do melhor modelo
12. Previsão numa imagem individual

> Funciona localmente (VS Code/Jupyter) ou no Google Colab: só tens de ajustar o `base_dir` na secção 2.0. O EfficientNet é pesado — com GPU (ou no Colab) é muito mais rápido.
>
> Uso exclusivamente educativo — não serve para diagnóstico clínico.

## 1. Instalação, imports e configuração
Se corres o notebook **localmente**, as bibliotecas têm de estar instaladas no mesmo ambiente Python do *kernel*. A primeira célula instala-as (só uma vez; depois reinicia o kernel). No Google Colab já vem tudo instalado.

In [ ]:
# Localmente, na primeira vez: muda para True, corre a célula e depois reinicia o kernel.
# Depois de instalado, volta a deixar False. (No Google Colab não é preciso.)
INSTALAR = False

if INSTALAR:
    %pip install -q tensorflow opencv-python scikit-learn seaborn matplotlib pandas tqdm pillow

import sys
print(sys.version)   # TensorFlow só tem pacotes para certas versões de Python (normalmente 3.9–3.12)

### 1.1 Imports
Só o que o notebook realmente usa: NumPy/Pandas, OpenCV, Matplotlib/Seaborn, scikit-learn (métricas e pesos de classe) e TensorFlow/Keras.

In [ ]:
import os
import gc
import time
import random
from types import SimpleNamespace

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from IPython.display import display
from sklearn.metrics import confusion_matrix, classification_report, f1_score, recall_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import Model, Sequential
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout,
                                     BatchNormalization, GlobalMaxPooling2D, Rescaling,
                                     RandomFlip, RandomRotation, RandomZoom)
from tensorflow.keras.applications import EfficientNetB0, EfficientNetB4
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

print('TensorFlow', tf.__version__)

### 1.2 Configuração global e *seeds*
- `SEED` fixa a aleatoriedade (NumPy, Python e TensorFlow) para as experiências serem repetíveis e comparáveis.
- `LIMIT_PER_CLASS` permite carregar só algumas imagens por classe, para **testar o pipeline todo em poucos minutos**. Deixa `None` para usar o dataset completo.
- `PRETRAINED` define os pesos do EfficientNet (`'imagenet'` normalmente, o que exige internet na primeira vez para descarregar os pesos; `None` serve só para testes do código, e dá um modelo sem pré-treino).
- `CACHE_TO_DISK` guarda em `cache/` os arrays já pré-processados de cada pipeline, para o pré-processamento (que pode demorar) só correr **uma vez**. Se alterares o código de um passo de pré-processamento, sobe o `CACHE_VERSION` para invalidar a cache antiga.

In [ ]:
SEED = 42
LIMIT_PER_CLASS = None    # ex.: 30 para um teste rápido; None = dataset completo
PRETRAINED = 'imagenet'   # pesos pré-treinados do EfficientNet
MODELS_DIR = 'models'     # onde os modelos treinados são guardados
CACHE_TO_DISK = True      # guarda os dados pré-processados em cache/ (cada pipeline ocupa ~0,5 GB a 128 px)
CACHE_DIR = 'cache'
CACHE_VERSION = 1         # sobe este número se alterares as funções de pré-processamento
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

def set_seed(seed=SEED):
    tf.keras.utils.set_random_seed(seed)   # fixa random, numpy e tensorflow de uma vez

set_seed()

## 2. Dataset
O código precisa das imagens organizadas por classe: `Datasets/training/<classe>/` e `Datasets/validation/<classe>/`.

### 2.0 Organizar o dataset em pastas (só na primeira vez)
O HAM10000 vem como uma pasta com imagens soltas e um ficheiro `HAM10000_metadata.csv`. Esta célula lê o CSV (coluna `dx` = diagnóstico), procura as imagens `.jpg` dentro de `base_dir` e copia-as para as pastas certas. A divisão 80/20 é feita **por lesão** (`lesion_id`): no HAM10000 a mesma lesão aparece em várias fotos, e se ficassem fotos da mesma lesão em treino e validação os resultados ficariam inflacionados (*data leakage*).

> `base_dir` tem de ser a pasta onde estão o CSV **e** as imagens. Se as pastas já estiverem organizadas, a célula não faz nada. Se a cópia for interrompida, apaga a pasta `Datasets` e corre outra vez.

In [ ]:
import shutil
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

# r'...' (raw string) evita o erro de "\U" nos caminhos do Windows
base_dir = Path(r'C:\Users\annad\Desktop\AAB')   # pasta com o CSV e as imagens
dataset_dir = base_dir / 'Datasets'                  # pasta que vai ser criada

if (dataset_dir / 'training').exists():
    print('Dataset já organizado em', dataset_dir)
else:
    metadata = pd.read_csv(base_dir / 'HAM10000_metadata.csv')

    # Procurar todas as imagens .jpg (ignorando a pasta de destino)
    paths = {f.stem: f for f in base_dir.rglob('*.jpg') if dataset_dir not in f.parents}
    metadata['path'] = metadata['image_id'].map(paths)
    print(f"Imagens encontradas: {metadata['path'].notna().sum()} de {len(metadata)}")
    metadata = metadata.dropna(subset=['path']).reset_index(drop=True)
    assert len(metadata) > 0, "Nenhuma imagem encontrada: confirma o base_dir e se as imagens estão lá."

    # Divisão 80/20 por lesão, para não haver fotos da mesma lesão nos dois conjuntos
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, val_idx = next(gss.split(metadata, groups=metadata['lesion_id']))

    for split, idx in [('training', train_idx), ('validation', val_idx)]:
        for _, row in tqdm(metadata.iloc[idx].iterrows(), total=len(idx), desc=split):
            dest = dataset_dir / split / row['dx']
            dest.mkdir(parents=True, exist_ok=True)
            shutil.copy(row['path'], dest / row['path'].name)
    print('Concluído:', dataset_dir)

### 2.1 Caminhos e classes
Define as pastas de treino e validação e a lista de classes (`categories`), obtida a partir dos nomes das subpastas, por ordem alfabética (a ordem é sempre a mesma, e é ela que define o número de cada classe).

In [ ]:
dataset = str(dataset_dir)
train_folder = os.path.join(dataset, "training")
val_folder = os.path.join(dataset, "validation")

categories = sorted(os.listdir(train_folder))
num_classes = len(categories)
print(categories, num_classes)

## 3. Distribuição das classes
Número de imagens por classe em treino e validação (só conta ficheiros, não carrega imagens). Serve para detetar **desbalanceamento** — no HAM10000 a classe `nv` domina, o que afeta o treino e justifica o uso de pesos por classe e de métricas além da accuracy.

In [ ]:
def count_images(folder):
    return {c: len(os.listdir(os.path.join(folder, c))) for c in categories}

train_counts = count_images(train_folder)
val_counts = count_images(val_folder)
print('Treino:', sum(train_counts.values()), '| Validação:', sum(val_counts.values()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = plt.get_cmap('viridis')(np.linspace(0, 1, num_classes))
for ax, counts, title in zip(axes, [train_counts, val_counts], ['Treino', 'Validação']):
    ax.bar(list(counts.keys()), list(counts.values()), color=colors)
    ax.bar_label(ax.containers[0])
    ax.set_title(f'Distribuição das classes ({title})')
    ax.set_xlabel('Classe'); ax.set_ylabel('Nº de imagens')
plt.tight_layout()
plt.show()

## 4. Exemplos de imagens
Mostra uma imagem aleatória de cada classe, para confirmar visualmente que os dados estão corretos e perceber como as classes se parecem.

In [ ]:
def read_rgb(path):
    # imdecode + fromfile funciona mesmo com caminhos com acentos no Windows
    img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

fig = plt.figure(figsize=(15, 10))
rows = int(np.ceil(num_classes / 3))
for k, c in enumerate(categories):
    f = random.choice(os.listdir(os.path.join(train_folder, c)))
    ax = fig.add_subplot(rows, 3, k + 1)
    ax.imshow(read_rgb(os.path.join(train_folder, c, f)))
    ax.set_title(c, pad=10, size=14)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Pré-processamento: passos, carregamento e visualização
Cada método de pré-processamento é um **passo** com nome. Uma *pipeline* é simplesmente uma sequência de passos, por exemplo `('hair', 'cc', 'rgb')`. Os passos são aplicados **pela ordem em que os indicas**, na imagem original (≈ 600×450), **antes** do redimensionamento. Os passos `'pad'` e `'rgb'` são especiais: o `'pad'` aplica-se mesmo antes de redimensionar, e o `'rgb'` no fim, sejam quais forem as posições em que os escrevas.

| Passo | O que faz | Porquê testá-lo (hipótese) |
|---|---|---|
| `rgb` | Converte BGR (OpenCV) para RGB | Os pesos do ImageNet foram treinados em RGB |
| `pad` | Margens pretas até a imagem ficar quadrada, em vez de a esticar | Evita deformar as lesões (as fotos são 4:3) |
| `hair` | Remove pelos: deteta estruturas escuras e finas (*blackhat* morfológico) e preenche-as por *inpainting* (DullRazor simplificado) | Os pelos tapam a lesão e podem confundir o modelo. Limitação: também pode apagar pequenas estruturas escuras reais |
| `cc` | *Color constancy* (Shades of Gray): corrige dominantes de cor da iluminação/câmara | As imagens vêm de equipamentos diferentes. Risco: a cor da pele e da lesão também é informação |
| `clahe` | Equalização adaptativa de contraste no canal de luminosidade (espaço LAB) | Realça bordas e pigmentação |
| `bright` | Correção gama (0.7) que "clareia" a imagem | Imagens escuras perdem detalhe |
| `border` | Se os cantos estão escuros (vinheta do dermatoscópio), recorta o centro | O fundo preto pode ser um atalho irrelevante para o modelo |
| `center` | Recorte central (80%) | A lesão costuma estar ao centro; ganha-se resolução efetiva. Risco: cortar lesões grandes |
| `gray` | Escala de cinzentos | **Controlo negativo**: mede quanto a cor importa |

**Decisão de memória:** os arrays ficam em `uint8` (0–255), sem dividir por 255. Em `float32` o dataset a 224×224 ocuparia cerca de 6 GB; em `uint8` ocupa cerca de 1,5 GB. A normalização é feita **dentro dos modelos**.

### 5.1 Os passos de pré-processamento
Cada função recebe e devolve uma imagem BGR `uint8`.

In [ ]:
def remove_hair(img, thresh=10):
    # DullRazor simplificado: blackhat morfológico deteta estruturas escuras e finas (pelos);
    # depois preenche-as por inpainting. O kernel escala com o tamanho da imagem (~17 px em 600x450).
    h, w = img.shape[:2]
    k = max(3, int(max(h, w) / 35)) | 1
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_RECT, (k, k)))
    _, mask = cv2.threshold(blackhat, thresh, 255, cv2.THRESH_BINARY)
    mask = cv2.dilate(mask, np.ones((3, 3), np.uint8))
    return cv2.inpaint(img, mask, 3, cv2.INPAINT_TELEA)

def color_constancy(img, power=6):
    # Shades of Gray: estima a cor do iluminante (norma de Minkowski) e corrige cada canal
    x = img.astype(np.float64)
    rgb = np.power(np.mean(np.power(x, power), axis=(0, 1)), 1.0 / power)
    rgb = rgb / np.sqrt(np.sum(rgb ** 2))
    gain = 1.0 / (np.sqrt(3) * rgb)
    return np.clip(x * gain, 0, 255).astype(np.uint8)

def clahe_lab(img, clip=2.0, tile=8):
    # CLAHE só no canal L (luminosidade), para não distorcer as cores
    l, a, b = cv2.split(cv2.cvtColor(img, cv2.COLOR_BGR2LAB))
    l = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile)).apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)

def brighten(img, gamma=0.7):
    # gamma < 1 clareia a imagem
    table = (((np.arange(256) / 255.0) ** gamma) * 255).astype(np.uint8)
    return cv2.LUT(img, table)

def crop_center(img, frac=0.8):
    h, w = img.shape[:2]
    ch, cw = int(h * frac), int(w * frac)
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    return img[y0:y0 + ch, x0:x0 + cw]

def remove_border(img, dark=40, keep=0.65):
    # só atua em imagens com vinheta escura (≥ 3 cantos escuros): recorta o centro, que fica sem fundo preto
    h, w = img.shape[:2]
    s = max(8, int(0.08 * min(h, w)))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    corners = [gray[:s, :s], gray[:s, -s:], gray[-s:, :s], gray[-s:, -s:]]
    if sum(c.mean() < dark for c in corners) >= 3:
        return crop_center(img, keep)
    return img

def to_gray(img):
    return cv2.cvtColor(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), cv2.COLOR_GRAY2BGR)

STEP_FUNCS = {'hair': remove_hair, 'cc': color_constancy, 'clahe': clahe_lab, 'bright': brighten,
              'border': remove_border, 'center': crop_center, 'gray': to_gray}
VALID_STEPS = set(STEP_FUNCS) | {'rgb', 'pad'}

def check_steps(steps):
    bad = [s for s in steps if s not in VALID_STEPS]
    if bad:
        raise ValueError(f'Passos desconhecidos: {bad}. Válidos: {sorted(VALID_STEPS)}')

### 5.2 Carregar imagens e conjuntos de dados
`load_image` aplica uma pipeline a **uma** imagem; `load_split` faz isso para uma pasta inteira; `get_data` junta treino e validação, guarda o resultado em memória e (se `CACHE_TO_DISK`) em disco. O pré-processamento é aplicado **igual** a treino e validação.

In [ ]:
def load_image(path, img_size=128, steps=()):
    img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)   # BGR, uint8
    if img is None:
        return None
    for s in steps:                       # passos "normais", pela ordem indicada
        if s in STEP_FUNCS:
            img = STEP_FUNCS[s](img)
    if 'pad' in steps:                    # tornar quadrada com margens pretas, sem deformar a lesão
        h, w = img.shape[:2]
        s = max(h, w)
        top, left = (s - h) // 2, (s - w) // 2
        img = cv2.copyMakeBorder(img, top, s - h - top, left, s - w - left,
                                 cv2.BORDER_CONSTANT, value=(0, 0, 0))
    img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_AREA)
    if 'rgb' in steps:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def load_split(folder, img_size=128, steps=()):
    X, Y = [], []
    for idx, c in enumerate(categories):
        cdir = os.path.join(folder, c)
        files = sorted(f for f in os.listdir(cdir) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
        if LIMIT_PER_CLASS:
            files = files[:LIMIT_PER_CLASS]
        for f in tqdm(files, desc=f'{os.path.basename(folder)}/{c}', leave=False):
            img = load_image(os.path.join(cdir, f), img_size, steps)
            if img is not None:
                X.append(img)
                Y.append(idx)
    return np.asarray(X, dtype=np.uint8), np.asarray(Y, dtype=np.int64)

_DATA_CACHE = {}
def get_data(img_size=128, steps=()):
    steps = tuple(steps)
    check_steps(steps)
    key = (img_size, steps, LIMIT_PER_CLASS)
    if key not in _DATA_CACHE:
        _DATA_CACHE.clear()   # só mantém uma configuração em memória
        gc.collect()
        use_disk = CACHE_TO_DISK and not LIMIT_PER_CLASS
        path = os.path.join(CACHE_DIR, f"v{CACHE_VERSION}_{img_size}_{'+'.join(steps) or 'base'}.npz")
        if use_disk and os.path.exists(path):
            d = np.load(path)
            _DATA_CACHE[key] = (d['X_tr'], d['Y_tr'], d['X_va'], d['Y_va'])
            print('Carregado da cache:', path)
        else:
            X_tr, Y_tr = load_split(train_folder, img_size, steps)
            X_va, Y_va = load_split(val_folder, img_size, steps)
            _DATA_CACHE[key] = (X_tr, Y_tr, X_va, Y_va)
            if use_disk:
                np.savez(path, X_tr=X_tr, Y_tr=Y_tr, X_va=X_va, Y_va=Y_va)
    return _DATA_CACHE[key]

### 5.3 Ver o efeito de cada pré-processamento
**Passo essencial para o relatório:** mostra a mesma imagem antes e depois de cada pipeline. Serve para confirmar que cada passo faz o que se espera (por exemplo, que a remoção de pelos não apaga partes da lesão) e para ilustrar o estudo. Volta a correr a célula para ver outras imagens (ou usa `classes=` para escolher as classes e `seed=` para fixar a escolha). A imagem original aparece só redimensionada.

In [ ]:
PIPELINES = {
    'base (rgb)':     ('rgb',),
    'sem pelos':      ('hair', 'rgb'),
    'color constancy': ('cc', 'rgb'),
    'CLAHE':          ('clahe', 'rgb'),
    'brilho (gama)':  ('bright', 'rgb'),
    'recorte borda':  ('border', 'rgb'),
    'recorte centro': ('center', 'rgb'),
    'cinzento':       ('gray', 'rgb'),
    'pelos + cc':     ('hair', 'cc', 'rgb'),
}

def show_pipelines(pipelines, classes=('mel', 'bkl'), size=300, seed=None):
    rng = random.Random(seed)
    names = ['original'] + list(pipelines)
    fig, axes = plt.subplots(len(classes), len(names), figsize=(2.6 * len(names), 2.9 * len(classes)))
    axes = np.atleast_2d(axes)
    for r, c in enumerate(classes):
        cdir = os.path.join(val_folder, c)
        path = os.path.join(cdir, rng.choice(os.listdir(cdir)))
        for k, name in enumerate(names):
            if name == 'original':
                img = cv2.resize(read_rgb(path), (size, size), interpolation=cv2.INTER_AREA)
                title = f'original ({c})'
            else:
                steps = pipelines[name]
                img = load_image(path, size, steps)
                if 'rgb' not in steps:
                    img = img[..., ::-1]    # para mostrar em RGB
                title = name
            axes[r, k].imshow(img)
            axes[r, k].axis('off')
            axes[r, k].set_title(title, fontsize=9)
    plt.tight_layout()
    plt.show()

show_pipelines(PIPELINES, classes=('mel', 'bkl'))

### 5.4 Carregar a configuração base
Carrega treino e validação com a configuração base (128×128, sem passos, ou seja, BGR e sem padding). É também a que o baseline usa. Confirma que as formas e os totais fazem sentido.

In [ ]:
X_train, Y_train, X_val, Y_val = get_data(128)
print('X_train:', X_train.shape, X_train.dtype, '| Y_train:', Y_train.shape)
print('X_val  :', X_val.shape, X_val.dtype, '| Y_val  :', Y_val.shape)
print('Intervalo de pixels:', X_train.min(), '-', X_train.max())

## 6. Funções auxiliares
Reutilizadas em todas as experiências:
- `plot_history` — curvas de accuracy e loss (treino vs validação), para detetar *overfitting*.
- `evaluate_model` — calcula accuracy, **macro F1**, **macro recall** e recall por classe na validação.
- `plot_confusion_matrix` — matriz de confusão (em contagens e normalizada por linha) e relatório de classificação.
- **Registo de resultados** — cada experiência acrescenta uma linha a `RESULTS` (e a `results.csv`, que sobrevive a reinícios do kernel). Cada linha guarda o **pipeline** (nome da experiência), os **passos** de pré-processamento, a *seed* e o **tipo** (`referência`, `controlo` ou `pré-processamento`). `show_summary()` agrega as repetições (média ± desvio-padrão por pipeline) e `impact_table()` mostra o efeito de cada pipeline face a uma referência.

In [ ]:
def plot_history(history, title):
    h = history.history
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(h['accuracy'], label='train_accuracy')
    plt.plot(h['val_accuracy'], label='val_accuracy')
    plt.xlabel('Epochs'); plt.ylabel('Accuracy'); plt.legend()
    plt.title(f'{title} Accuracy Curves')

    plt.subplot(1, 2, 2)
    plt.plot(h['loss'], label='train_loss')
    plt.plot(h['val_loss'], label='val_loss')
    plt.xlabel('Epochs'); plt.ylabel('Loss'); plt.legend()
    plt.title(f'{title} Loss Curves')

    plt.tight_layout()
    plt.show()

def evaluate_model(model, X, Y):
    probs = model.predict(X, batch_size=64, verbose=0)
    y_pred = probs.argmax(axis=1)
    labels = list(range(num_classes))
    recalls = recall_score(Y, y_pred, labels=labels, average=None, zero_division=0)
    metrics = {
        'accuracy': float((y_pred == Y).mean()),
        'macro_f1': float(f1_score(Y, y_pred, labels=labels, average='macro', zero_division=0)),
        'macro_recall': float(recalls.mean()),
        'val_loss': float(np.mean(tf.keras.losses.sparse_categorical_crossentropy(Y, probs).numpy())),
    }
    for c, r in zip(categories, recalls):
        metrics[f'recall_{c}'] = float(r)
    return metrics

def plot_confusion_matrix(model, X, Y, categories, title):
    y_pred = model.predict(X, batch_size=64, verbose=0).argmax(axis=1)
    labels = list(range(len(categories)))
    cm = confusion_matrix(Y, y_pred, labels=labels)
    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=categories, yticklabels=categories, ax=axes[0])
    axes[0].set_title(f'{title} (contagens)')
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=categories, yticklabels=categories,
                vmin=0, vmax=1, ax=axes[1])
    axes[1].set_title(f'{title} (normalizada por linha = recall)')
    for ax in axes:
        ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.show()

    print("\n Classification Report:\n")
    print(classification_report(Y, y_pred, labels=labels, target_names=categories, zero_division=0))

In [ ]:
RESULTS_CSV = 'results.csv'

def _isna(x):
    return x is None or (isinstance(x, float) and np.isnan(x))

def _fix_row(r):
    # compatibilidade com linhas antigas (anteriores aos campos steps / pipeline / seed / kind)
    if _isna(r.get('steps')):
        flags = [s for s in ('rgb', 'pad') if str(r.get(s)).lower() == 'true']
        r['steps'] = '+'.join(flags)
    if _isna(r.get('pipeline')):
        r['pipeline'] = r['name']
    if _isna(r.get('seed')):
        r['seed'] = SEED
    if _isna(r.get('kind')):
        n = str(r['name'])
        r['kind'] = ('baseline' if n.startswith('CNN') else
                     'referência' if n.startswith('E1_') else
                     'controlo' if n.split('_')[0] in ('E4', 'E5') else 'pré-processamento')
    return r

RESULTS = ([_fix_row(r) for r in pd.read_csv(RESULTS_CSV).to_dict('records')]
           if os.path.exists(RESULTS_CSV) else [])

MAIN_COLS = ['name', 'kind', 'steps', 'variant', 'img_size', 'augment', 'class_weights', 'fine_tune',
             'accuracy', 'macro_f1', 'macro_recall', 'recall_mel', 'epochs', 'minutes']

def _save_results():
    pd.DataFrame(RESULTS).to_csv(RESULTS_CSV, index=False)

def add_result(row):
    global RESULTS
    RESULTS = [r for r in RESULTS if r['name'] != row['name']] + [row]   # voltar a correr = substitui
    _save_results()

def remove_result(name):
    global RESULTS
    RESULTS = [r for r in RESULTS if r['name'] != name]
    _save_results()

def results_table():
    df = pd.DataFrame(RESULTS)
    if df.empty:
        return df
    return df.sort_values('macro_f1', ascending=False).reset_index(drop=True)

def show_results():
    df = results_table()
    if df.empty:
        print('Ainda não há resultados.')
        return
    df = df[MAIN_COLS].copy()
    df['steps'] = df['steps'].replace('', '(nenhum)')
    display(df.round(3))

def summary_by_pipeline():
    # agrega as repetições (seeds) de cada pipeline: média e desvio-padrão
    df = pd.DataFrame(RESULTS)
    if df.empty:
        return df
    s = df.groupby('pipeline').agg(
        kind=('kind', 'first'), steps=('steps', 'first'), img_size=('img_size', 'first'),
        n_seeds=('seed', 'nunique'), accuracy=('accuracy', 'mean'),
        macro_f1=('macro_f1', 'mean'), macro_f1_std=('macro_f1', 'std'),
        macro_recall=('macro_recall', 'mean'), macro_recall_std=('macro_recall', 'std'),
        recall_mel=('recall_mel', 'mean'), recall_mel_std=('recall_mel', 'std'))
    return s.sort_values('macro_f1', ascending=False)

def show_summary():
    s = summary_by_pipeline()
    if s.empty:
        print('Ainda não há resultados.')
        return
    s = s.copy()
    s['steps'] = s['steps'].replace('', '(nenhum)')
    display(s.round(3))

def impact_table(ref='E2_rgb'):
    # efeito de cada pipeline face à referência (diferença nas métricas médias)
    s = summary_by_pipeline()
    if s.empty or ref not in s.index:
        print(f"A referência '{ref}' ainda não tem resultados.")
        return
    out = s[['kind', 'steps', 'n_seeds', 'macro_f1', 'macro_f1_std', 'recall_mel']].copy()
    out['Δ macro_f1'] = out['macro_f1'] - s.loc[ref, 'macro_f1']
    out['Δ recall_mel'] = out['recall_mel'] - s.loc[ref, 'recall_mel']
    out['steps'] = out['steps'].replace('', '(nenhum)')
    display(out.drop(index=ref).sort_values('Δ macro_f1', ascending=False).round(3))

def plot_results():
    df = results_table()
    if df.empty:
        return
    ax = df.set_index('name')[['macro_f1', 'macro_recall', 'recall_mel']].plot.bar(figsize=(12, 5))
    ax.set_title('Comparação de experiências (validação) — cada corrida')
    ax.set_ylim(0, 1); ax.set_ylabel('Métrica'); ax.set_xlabel('')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

def plot_summary():
    s = summary_by_pipeline()
    if s.empty:
        return
    cols = ['macro_f1', 'macro_recall', 'recall_mel']
    ax = s[cols].plot.bar(figsize=(12, 5), yerr={c: s[c + '_std'].fillna(0) for c in cols}, capsize=3)
    ax.set_title('Média por pipeline (barras de erro = desvio-padrão entre seeds; sem barra = 1 seed)')
    ax.set_ylim(0, 1); ax.set_ylabel('Métrica'); ax.set_xlabel('')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

def register(name, model, X, Y, cfg, epochs, t0):
    # guarda o modelo, avalia-o e acrescenta uma linha à tabela de resultados
    path = os.path.join(MODELS_DIR, f'{name}.keras')
    model.save(path)
    row = {'name': name, **cfg, **evaluate_model(model, X, Y),
           'epochs': epochs, 'minutes': round((time.time() - t0) / 60, 1), 'model_path': path}
    add_result(row)
    return row

def analyze_run(name):
    # reabre a análise individual de uma corrida já feita: matriz de confusão + relatório por classe
    row = next((r for r in RESULTS if r['name'] == name), None)
    if row is None:
        print(f"Não há resultados para '{name}'. Corridas disponíveis:", [r['name'] for r in RESULTS])
        return
    steps = tuple(s for s in str(row['steps']).split('+') if s)
    model = tf.keras.models.load_model(os.path.join(MODELS_DIR, f'{name}.keras'))
    _, _, X, Y = get_data(int(row['img_size']), steps)
    plot_confusion_matrix(model, X, Y, categories, name)

## 7. Baseline — CNN básico
Modelo simples construído de raiz, usado como **referência** para comparar com o EfficientNet. Esta secção é opcional: podes saltá-la sem afetar o resto. Trabalha diretamente sobre os arrays `uint8` da secção 5.1, por isso a primeira camada é um `Rescaling(1/255)`.

No treino anterior (100 épocas, sem *early stopping*) o modelo chegava a accuracy de treino de 1.00 contra ≈ 0.71 em validação, ou seja, overfitting forte. Agora o treino pára sozinho quando a `val_accuracy` deixa de melhorar.

In [ ]:
def build_basic_cnn(input_shape, num_classes):
    model = Sequential([
        Input(shape=input_shape),
        Rescaling(1.0 / 255),
        Conv2D(32, (3, 3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(num_classes, activation='softmax'),
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

basic_cnn = build_basic_cnn((128, 128, 3), num_classes)
basic_cnn.summary()

### 7.1 Treino
Até 50 épocas, com *early stopping* na `val_accuracy` (paciência 10) e recuperação dos melhores pesos.

In [ ]:
set_seed()
t0 = time.time()
history_cnn = basic_cnn.fit(
    X_train, Y_train, validation_data=(X_val, Y_val),
    epochs=50, batch_size=32,
    callbacks=[EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)],
)
plot_history(history_cnn, 'Basic CNN')

### 7.2 Avaliação e registo
Calcula as métricas na validação, guarda o modelo e acrescenta a linha `CNN_basico` à tabela de resultados.

In [ ]:
cfg = dict(pipeline='CNN_basico', seed=SEED, kind='baseline', steps='', variant='CNN', img_size=128,
           augment=False, class_weights='none', fine_tune=False)
row = register('CNN_basico', basic_cnn, X_val, Y_val, cfg, len(history_cnn.history['loss']), t0)
print({k: round(v, 3) for k, v in row.items() if k in ('accuracy', 'macro_f1', 'macro_recall', 'recall_mel')})
plot_confusion_matrix(basic_cnn, X_val, Y_val, categories, 'Basic CNN')

## 8. EfficientNet — construção, treino e fine-tuning
Aqui ficam as funções; as experiências em si estão na secção 9.

**Transfer learning em duas fases:**
1. **Cabeça:** o EfficientNet pré-treinado no ImageNet fica **congelado** e só se treina a cabeça nova (`GlobalMaxPooling2D` → Dense(256) → BatchNorm → Dropout(0.5) → softmax).
2. **Fine-tuning (opcional):** descongelam-se as últimas `ft_layers` camadas do backbone (as `BatchNormalization` ficam sempre congeladas) e treina-se com *learning rate* muito baixa (1e-5). Foi o passo que faltou na primeira experiência, em que o backbone congelado levou a subajuste.

**Outras decisões:**
- **Data augmentation** (flips, rotação, zoom) só atua no treino. Não se alteram cores, porque a cor é informação clínica relevante.
- **Pesos por classe:** `balanced` dá à classe `df` um peso ≈ 50 vezes superior ao de `nv`, o que gerou muitos falsos alarmes. O modo `sqrt` (por defeito) usa a raiz quadrada desses pesos, mais suave.
- O EfficientNet do Keras **já inclui** a normalização de entrada e espera pixels 0–255, que é exatamente o que os arrays `uint8` fornecem.
- *Early stopping* e redução da *learning rate* na `val_loss`; no fim ficam os melhores pesos.

In [ ]:
def make_augmentation():
    return Sequential([
        RandomFlip('horizontal_and_vertical'),
        RandomRotation(0.2),
        RandomZoom(0.1),
    ], name='data_augmentation')

def build_efficientnet(variant='B4', img_size=128, augment=True, head_lr=5e-4):
    Base = {'B0': EfficientNetB0, 'B4': EfficientNetB4}[variant]
    base = Base(include_top=False, weights=PRETRAINED, input_shape=(img_size, img_size, 3))
    base.trainable = False   # fase 1: só a cabeça aprende

    inputs = Input(shape=(img_size, img_size, 3))
    x = make_augmentation()(inputs) if augment else inputs
    x = base(x, training=False)
    x = GlobalMaxPooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(head_lr), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model, base

def unfreeze_top_layers(model, base, n_layers=30, lr=1e-5):
    base.trainable = True
    for layer in base.layers[:-n_layers]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, BatchNormalization):
            layer.trainable = False   # mantém as estatísticas do ImageNet
    model.compile(optimizer=Adam(lr), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

def get_class_weights(y, mode='sqrt'):
    assert mode in ('none', 'sqrt', 'balanced')
    if mode == 'none':
        return None
    classes = np.unique(y)
    cw = compute_class_weight('balanced', classes=classes, y=y)
    if mode == 'sqrt':
        cw = np.sqrt(cw)
    return {int(c): float(w) for c, w in zip(classes, cw)}

def merge_history(*hs):
    keys = hs[0].history.keys()
    return SimpleNamespace(history={k: sum((list(h.history[k]) for h in hs), []) for k in keys})

### 8.1 `run_experiment`
Uma única função que faz tudo: carrega os dados com a pipeline de pré-processamento pedida (`steps`), constrói o modelo, treina (cabeça + fine-tuning opcional), mostra as curvas, avalia, guarda o modelo e regista uma linha na tabela de resultados. Basta mudar **um parâmetro de cada vez**.

- `name` é o nome do **pipeline**. Com `seed` diferente da `SEED` base, a corrida guarda-se como `name_s<seed>`, mas conta como repetição do mesmo pipeline.
- `kind` classifica a experiência: `'referência'`, `'controlo'` (muda algo do treino que **não** é pré-processamento) ou `'pré-processamento'`.
- **Cada chamada treina um modelo novo, de raiz, com os dados de UMA só pipeline** (os dados das outras não entram). No fim mostra as curvas e, com `show_report=True` (por defeito), a matriz de confusão e as métricas por classe dessa corrida. Mais tarde podes reabrir esta análise com `analyze_run('E7_hair')`.

In [ ]:
def run_experiment(name, steps=(), variant='B4', img_size=128, augment=True,
                   class_weight_mode='sqrt', epochs_head=15, fine_tune=True, epochs_ft=15,
                   ft_layers=30, batch_size=32, seed=SEED, kind='pré-processamento', show_report=True):
    steps = tuple(steps)
    run_name = name if seed == SEED else f'{name}_s{seed}'
    print(f"\n=== {run_name} | passos: {'+'.join(steps) or '(nenhum)'} | seed {seed} ===")
    tf.keras.backend.clear_session()
    gc.collect()
    set_seed(seed)
    t0 = time.time()

    X_tr, Y_tr, X_va, Y_va = get_data(img_size, steps)
    cw = get_class_weights(Y_tr, class_weight_mode)
    if cw:
        print('Pesos por classe:', {categories[k]: round(v, 2) for k, v in cw.items()})

    def callbacks():
        return [EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
                ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, verbose=1)]

    model, base = build_efficientnet(variant, img_size, augment)
    hists = [model.fit(X_tr, Y_tr, validation_data=(X_va, Y_va), epochs=epochs_head,
                       batch_size=batch_size, class_weight=cw, callbacks=callbacks())]
    if fine_tune:
        unfreeze_top_layers(model, base, ft_layers)
        hists.append(model.fit(X_tr, Y_tr, validation_data=(X_va, Y_va), epochs=epochs_ft,
                               batch_size=batch_size, class_weight=cw, callbacks=callbacks()))

    history = merge_history(*hists)
    plot_history(history, run_name)

    cfg = dict(pipeline=name, seed=seed, kind=kind, steps='+'.join(steps), variant=variant,
               img_size=img_size, augment=augment, class_weights=class_weight_mode, fine_tune=fine_tune)
    row = register(run_name, model, X_va, Y_va, cfg, len(history.history['loss']), t0)
    print({k: round(v, 3) for k, v in row.items() if k in ('accuracy', 'macro_f1', 'macro_recall', 'recall_mel')},
          f"| {row['minutes']} min")
    if show_report:   # análise individual desta corrida: matriz de confusão e métricas por classe
        plot_confusion_matrix(model, X_va, Y_va, categories, run_name)
    return row

## 9. Experiências
**Pergunta do estudo:** qual é o impacto de cada método de pré-processamento no EfficientNetB4?

**Para que a resposta seja válida**, tudo o resto tem de estar fixo. Por isso as experiências dividem-se em três fases, e é importante saber **porque é que cada uma existe**:

- **Fase A — Controlo (não são pré-processamento).** Servem para escolher e justificar a **configuração de treino** que fica fixa no resto do estudo. Se a configuração de treino variasse entre experiências, não se saberia se uma diferença vem do pré-processamento ou do treino.
- **Fase B — Pré-processamento (o objeto do estudo).** Cada experiência acrescenta **um** passo à linha de base simples (`E2`: redimensionar + RGB).
- **Fase C — Repetições.** Com *seeds* diferentes, para medir quanto os resultados variam só por acaso e saber se uma diferença é real.

| ID | Fase / tipo | O que muda | Porque está aqui (hipótese) |
|---|---|---|---|
| E1 | A — referência | 128×128, BGR, sem passos | Ponto de partida do estudo |
| E4 | A — controlo | Sem data augmentation | A augmentation altera as imagens no treino; vê-se se interfere com o efeito do pré-processamento |
| E5 | A — controlo | Pesos de classe `balanced` | O HAM10000 é muito desbalanceado (`nv` ≈ 67%); o tratamento do desbalanceamento condiciona as métricas macro |
| E2 | B — **linha de base** | `rgb` | Redimensionar + RGB é o pré-processamento mínimo; os pesos do ImageNet esperam RGB |
| E3 | B | `pad` | Não deformar as lesões |
| E6 | B | resolução 224 | Redimensionar menos reduz a perda de detalhe |
| E7 | B | `hair` | Os pelos ocultam a lesão |
| E8 | B | `cc` | Reduz diferenças de iluminação/câmara |
| E9 | B | `clahe` | Realça contraste local |
| E10 | B | `bright` | "Clarear" imagens escuras |
| E11 | B | `border` | Remove a vinheta escura do dermatoscópio |
| E12 | B | `center` | Aumenta a resolução efetiva na lesão |
| E13 | B — controlo negativo | `gray` | Mede quanto a cor é usada pelo modelo |
| E14 | B | `hair` + `cc` | Efeito combinado de dois passos |

**Notas para a interpretação**
- E3 e E6 foram corridas sobre BGR (como a E1), enquanto E7–E14 usam RGB (como a E2). Para uma comparação estritamente justa, podem repetir-se E3 e E6 com `'rgb'` (opcional).
- Nos testes até agora o ruído de amostragem da validação foi de cerca de **±0.04** no macro recall. Diferenças menores só se podem afirmar com repetições (Fase C).
- Decide pelo **macro F1** e pelo **recall de `mel`**, não pela accuracy.

### 9.0 Teste rápido do pipeline (recomendado)
Antes das corridas completas, testa o fluxo todo com 30 imagens por classe, o EfficientNetB0, **vários passos de pré-processamento** e 1 época. Demora poucos minutos e apanha erros de caminhos, memória, bibliotecas ou de pré-processamento. No fim (mesmo que dê erro), repõe o dataset completo e apaga o teste da tabela.

> **Precisa de internet na primeira vez:** os pesos do ImageNet de cada variante (B0, B4) são descarregados de `storage.googleapis.com` e ficam guardados em `~/.keras/models`. Se aparecer `URL fetch failure ... getaddrinfo failed`, é falta de ligação/DNS (VPN, proxy, firewall) e não um erro do código.

In [ ]:
LIMIT_PER_CLASS = 30
try:
    run_experiment('TESTE_RAPIDO', steps=('hair', 'cc', 'clahe', 'rgb'), variant='B0', epochs_head=1, fine_tune=False)
finally:
    # corre sempre, mesmo que a experiência dê erro: volta ao dataset completo e limpa o teste
    LIMIT_PER_CLASS = None
    remove_result('TESTE_RAPIDO')

### Fase A — Controlo da configuração de treino
> Se já correste estas experiências, **não precisas de as repetir**: os resultados estão em `results.csv` (certifica-te de que o notebook corre na mesma pasta).

#### E1 — Referência (128×128, BGR, sem passos)
> Dica: para medir o tempo antes de uma corrida longa, usa `epochs_head=2, epochs_ft=2` numa primeira vez.

In [ ]:
run_experiment('E1_B4_128_ref', kind='referência')

#### E4 — Sem data augmentation (controlo)

In [ ]:
run_experiment('E4_sem_aug', augment=False, kind='controlo')

#### E5 — Pesos de classe `balanced` (controlo)
Pesos mais agressivos do que o `sqrt` usado por defeito.

In [ ]:
run_experiment('E5_cw_balanced', class_weight_mode='balanced', kind='controlo')

### Fase B — Pré-processamento
Configuração de treino fixa (B4, 128×128, augmentation, pesos `sqrt`, fine-tuning); só os `steps` mudam. Antes de correres cada uma, vê o efeito do passo na secção 5.3.

#### E2 — Linha de base: redimensionar + RGB

In [ ]:
run_experiment('E2_rgb', steps=('rgb',))

#### E3 — Padding (sem deformar)

In [ ]:
run_experiment('E3_pad', steps=('pad',))

#### E6 — Resolução 224×224
**Pesada:** ≈ 1,5 GB de RAM só para as imagens e várias vezes mais tempo por época. Recomenda-se o Colab com GPU.

In [ ]:
run_experiment('E6_224', img_size=224)

#### E7 — Remoção de pelos
A primeira vez demora mais (o pré-processamento corre em ≈ 10 000 imagens; depois fica em cache).

In [ ]:
run_experiment('E7_hair', steps=('hair', 'rgb'))

#### E8 — Color constancy (Shades of Gray)

In [ ]:
run_experiment('E8_color_constancy', steps=('cc', 'rgb'))

#### E9 — CLAHE (contraste local)

In [ ]:
run_experiment('E9_clahe', steps=('clahe', 'rgb'))

#### E10 — Clarear (correção gama)

In [ ]:
run_experiment('E10_brilho', steps=('bright', 'rgb'))

#### E11 — Recorte da vinheta escura

In [ ]:
run_experiment('E11_borda', steps=('border', 'rgb'))

#### E12 — Recorte central (80%)

In [ ]:
run_experiment('E12_centro', steps=('center', 'rgb'))

#### E13 — Escala de cinzentos (controlo negativo)
Se o desempenho cair muito, a cor é importante para o modelo; se cair pouco, o modelo usa sobretudo forma e textura.

In [ ]:
run_experiment('E13_cinzento', steps=('gray', 'rgb'))

#### E14 — Combinação: pelos + color constancy

In [ ]:
run_experiment('E14_hair_cc', steps=('hair', 'cc', 'rgb'))

### Fase C — Repetições com várias *seeds*
Diferenças pequenas entre pipelines podem ser só ruído do treino. Repete a **linha de base** e os **pipelines mais promissores** (ou duvidosos) com mais *seeds*; a corrida original conta como a *seed* 42. Com 3 *seeds* por pipeline já podes comparar médias e desvios-padrão (secção 10). Cada corrida repete o treino completo, por isso escolhe poucos pipelines.

In [ ]:
# Exemplo: linha de base e dois pipelines (ajusta os passos aos que te interessarem)
REPETIR = {
    'E2_rgb':  ('rgb',),
    'E7_hair': ('hair', 'rgb'),
    'E8_color_constancy': ('cc', 'rgb'),
}
for seed in (1, 2):
    for nome, passos in REPETIR.items():
        run_experiment(nome, steps=passos, seed=seed)

## 10. Comparação de resultados
- `show_results()`: uma linha por corrida.
- `show_summary()`: uma linha por **pipeline** (média por *seeds*; o desvio-padrão só existe com 2 ou mais).
- `impact_table()`: efeito de cada pipeline **face à referência** (Δ do macro F1 e do recall de `mel`).
- `analyze_run('nome')`: reabre a análise individual (matriz de confusão e relatório por classe) de qualquer corrida já feita.

O macro F1 é a média do F1 de todas as classes (dá o mesmo peso às classes raras). Os resultados ficam também em `results.csv`.

> Nota: as métricas são calculadas na pasta de validação, que também é usada para escolher o melhor modelo (via *early stopping* e comparação de experiências). Isto torna os números ligeiramente **otimistas**, e diferenças pequenas (1–2 pontos) podem ser só ruído, sobretudo em classes pequenas como `df` (18 imagens). Para um resultado final rigoroso, deve reservar-se um conjunto de teste à parte.

In [ ]:
show_results()
plot_results()

In [ ]:
show_summary()
plot_summary()
impact_table(ref='E2_rgb')

## 11. Análise do melhor modelo
Carrega o modelo com maior macro F1, volta a preparar a validação **com o mesmo pré-processamento** com que ele foi treinado e mostra a matriz de confusão e o relatório de classificação.

In [ ]:
best = results_table().iloc[0]
print(best[MAIN_COLS].to_string())

best_cfg = dict(img_size=int(best['img_size']), steps=tuple(s for s in str(best['steps']).split('+') if s))
best_model = tf.keras.models.load_model(best['model_path'])
_, _, X_v, Y_v = get_data(best_cfg['img_size'], best_cfg['steps'])
plot_confusion_matrix(best_model, X_v, Y_v, categories, str(best['name']))

## 12. Previsão numa imagem individual
Escolhe uma imagem aleatória da pasta de validação (de uma classe à escolha, ou aleatória), aplica **o mesmo pré-processamento do melhor modelo**, e mostra a imagem com as 3 classes mais prováveis.

> As percentagens vêm do *softmax* e **não são probabilidades calibradas**: um modelo pode estar errado com confiança alta.

In [ ]:
def predict_random_image(model, cfg, classe=None, top_k=3):
    classe = classe or random.choice(categories)
    cdir = os.path.join(val_folder, classe)
    path = os.path.join(cdir, random.choice(os.listdir(cdir)))

    img = load_image(path, cfg['img_size'], cfg['steps'])
    probs = model.predict(img[None, ...], verbose=0)[0]
    top = np.argsort(probs)[::-1][:top_k]

    print('Real Class:', classe)
    for i in top:
        print(f'  {categories[i]:<6} {probs[i] * 100:5.1f}%')

    plt.imshow(read_rgb(path))
    plt.title(f'Real: {classe} | Predicted: {categories[top[0]]}')
    plt.axis('off')
    plt.show()

predict_random_image(best_model, best_cfg)               # classe aleatória
# predict_random_image(best_model, best_cfg, classe='mel')   # ou escolhe uma classe